# Bronze Layer: Load Amazon Products from Azure Blob Storage

This notebook reads CSV data from Azure Blob Storage and loads it into the bronze layer as a Delta table.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import current_timestamp, lit
from datetime import datetime

print("=" * 80)
print("BRONZE LAYER INGESTION: Amazon Products")
print("=" * 80)
print(f"Start Time: {datetime.now()}")
print()

In [ ]:
# Configuration
storage_account_name = "fabricsatest123"
storage_account_key = "eyIW38VoXk6BQ8dYJG2FTiAdIEafvXA5btOq+oePQ2frm/WYvpEdUd21I1jjDKm9kPlCoO0cGIjZ+AStOcJPMw=="
container_name = "fabric"
file_path = "amazon.csv"
target_table = "bronze_amazon_products"

print(f"Source: wasbs://{container_name}@{storage_account_name}.blob.core.windows.net/{file_path}")
print(f"Target: {target_table}")
print()

In [ ]:
# Configure Blob Storage Access
print("Configuring blob storage access...")
spark.conf.set(
    f"fs.azure.account.key.{storage_account_name}.blob.core.windows.net",
    storage_account_key
)
print("✅ Blob storage configured")
print()

In [ ]:
# Read CSV from Blob Storage
blob_path = f"wasbs://{container_name}@{storage_account_name}.blob.core.windows.net/{file_path}"
print(f"Reading CSV file from: {blob_path}")

df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("multiLine", "true") \
    .option("escape", '"') \
    .load(blob_path)

row_count = df.count()
print(f"✅ Successfully read {row_count} rows")
print()

In [ ]:
# Display Schema and Sample Data
print("Schema:")
print("-" * 80)
df.printSchema()
print()

print("Sample Data (first 5 rows):")
print("-" * 80)
df.show(5, truncate=False)
print()

In [ ]:
# Add Audit Columns
print("Adding audit columns...")
df_bronze = df \
    .withColumn("ingestion_timestamp", current_timestamp()) \
    .withColumn("source_file", lit(file_path)) \
    .withColumn("source_system", lit("azure_blob_storage"))

print("✅ Audit columns added")
print()

In [ ]:
# Write to Bronze Layer
print(f"Writing to bronze layer: {target_table}")

df_bronze.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(target_table)

print(f"✅ Successfully wrote {row_count} rows to {target_table}")
print()

# Verification
print("Verifying data in bronze layer...")
df_verify = spark.table(target_table)
verify_count = df_verify.count()
print(f"✅ Verified: {verify_count} rows in {target_table}")
print()

print("=" * 80)
print("BRONZE LAYER INGESTION COMPLETE")
print("=" * 80)